# Walkthrough 2: from mini-batch to prediction, loss and one training step (Base-RelBench)

Continues [Walkthrough 1](01_data_graph_sampling.ipynb). We take one small mini-batch
(4 seeds, fan-out `[4, 2]`) through our actual model, `BaseModel` in
[`experiments/train.py`](../experiments/train.py), and print every intermediate result.
Two steps are recomputed **by hand** and checked against PyTorch/PyG.

| Step | What you see |
|---|---|
| 1 | Setup: graph, task, one mini-batch, the model |
| 2 | Column encoders: one numerical value encoded by hand |
| 3 | Row encoder (ResNet): one embedding per row |
| 4 | Relative-time encoding → `h⁽⁰⁾` |
| 5 | One GraphSAGE layer, recomputed by hand for one seed driver |
| 6 | Second layer and head → prediction |
| 7 | Loss, recomputed by hand |
| 8 | One optimiser step |
| 9 | Evaluation on the validation split |
| 10 | Regression: the clipping range |

In [1]:
import os, sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'walkthroughs' else Path.cwd()
os.environ.setdefault('HF_HOME', str(ROOT / 'data' / 'hf'))
sys.path.insert(0, str(ROOT / 'experiments'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch_frame
from relbench import load_dataset
from relbench.modeling.graph import get_node_train_table_input, make_pkey_fkey_graph
from relbench.modeling.utils import get_stype_proposal
from torch_geometric.loader import NeighborLoader
from torch_geometric.seed import seed_everything

from train import BaseModel  # our model (experiments/train.py)

torch.set_printoptions(precision=4, sci_mode=False, linewidth=140)
seed_everything(0)

## Step 1: setup

Same graph as Walkthrough 1 (text columns skipped), the `driver-top3` task, one small
mini-batch, and a freshly initialised `BaseModel` (128 channels, 2 layers, `aggr='sum'`).
If you re-run the notebook, exact numbers may differ slightly from the saved outputs
(neighbour sampling is random); every hand-vs-model check still matches.

In [2]:
dataset = load_dataset('rel-f1')
db = dataset.get_db()
task = dataset.load_task('driver-top3')
col_to_stype = {t: {c: s for c, s in cols.items() if s != torch_frame.text_embedded}
                for t, cols in get_stype_proposal(db).items()}
data, col_stats = make_pkey_fkey_graph(
    db, col_to_stype_dict=col_to_stype, text_embedder_cfg=None,
    cache_dir=str(ROOT / 'data/materialized/rel-f1/none'))

train_input = get_node_train_table_input(table=task.get_table('train'), task=task)
loader = NeighborLoader(
    data, num_neighbors=[4, 2], time_attr='time',
    input_nodes=train_input.nodes, input_time=train_input.time,
    transform=train_input.transform, batch_size=4, temporal_strategy='uniform', shuffle=True)
batch = next(iter(loader))

model = BaseModel(data, col_stats, channels=128, out_channels=1, num_layers=2, aggr='sum')
print('seed drivers:', batch['drivers'].n_id.tolist(), '| labels:', batch['drivers'].y.tolist())
print('parameters:', sum(p.numel() for p in model.parameters()))
print(model)

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

Fetching 24 files: 100%|██████████| 24/24 [00:00<00:00, 1041.43it/s]

seed drivers: [17, 112, 30, 68] | labels: [0.0, 0.0, 1.0, 0.0]
parameters: 4150401
BaseModel(
  (encoder): HeteroEncoder(
    (encoders): ModuleDict(
      (races): ResNet(
        (encoder): StypeWiseFeatureEncoder(
          (encoder_dict): ModuleDict(
            (categorical): EmbeddingEncoder(
              (emb): Embedding(61, 128, padding_idx=0)
            )
            (numerical): LinearEncoder()
            (timestamp): TimestampEncoder(
              (positional_encoding): PositionalEncoding()
              (cyclic_encoding): CyclicEncoding()
            )
          )
        )
        (backbone): Sequential(
          (0): FCResidualBlock(
            (lin1): Linear(in_features=512, out_features=128, bias=True)
            (lin2): Linear(in_features=128, out_features=128, bias=True)
            (relu): ReLU()
            (dropout): Dropout(p=0.2, inplace=False)
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
            (norm2): LayerN

## Step 2: column encoders

Each table has its own `torch_frame.nn.ResNet`. Its first part,
`StypeWiseFeatureEncoder`, turns every column into a 128-d vector according to the
column's type. A numerical value is standardised with the column's training statistics,
then scaled by a learned vector and shifted by a learned bias:

$$e_{v,c} = \frac{x_{v,c} - \mu_c}{\sigma_c}\,\mathbf w_c + \mathbf b_c \in \mathbb R^{128}$$

In [3]:
tf = batch['results'].tf
enc = model.encoder.encoders['results']        # torch_frame ResNet for `results`
x_cols, col_names = enc.encoder(tf)            # StypeWiseFeatureEncoder
print('results rows in batch:', tf.num_rows, '| columns:', col_names)
print('per-column embeddings:', tuple(x_cols.shape), '= [rows, columns, channels]')

# Recompute one numerical value by hand: column `laps`, first row
lin = enc.encoder.encoder_dict['numerical']    # LinearEncoder
num_cols = tf.col_names_dict[torch_frame.numerical]
j = num_cols.index('laps')
x = tf.feat_dict[torch_frame.numerical][0, j]
by_hand = (x - lin.mean[j]) / lin.std[j] * lin.weight[j] + lin.bias[j]
print(f"\nlaps = {x.item():.0f}, column mean {lin.mean[j]:.1f}, std {lin.std[j]:.1f}")
print('first 5 channels by hand :', by_hand[:5].detach())
print('first 5 channels by model:', x_cols[0, col_names.index('laps'), :5].detach())
assert torch.allclose(by_hand, x_cols[0, col_names.index('laps')], atol=1e-5)
print('match ✓')

results rows in batch: 12 | columns: ['fastestLap', 'grid', 'laps', 'milliseconds', 'number', 'points', 'position', 'positionOrder', 'rank', 'statusId', 'date']
per-column embeddings: (12, 11, 128) = [rows, columns, channels]

laps = 71, column mean 44.1, std 32.0
first 5 channels by hand : tensor([ 0.0113, -0.0074, -0.0006,  0.0021,  0.0077])
first 5 channels by model: tensor([ 0.0113, -0.0074, -0.0006,  0.0021,  0.0077])
match ✓


## Step 3: row encoder

The column vectors are concatenated (`11 columns × 128 = 1408` numbers for `results`),
then passed through 4 residual blocks (Linear → LayerNorm → ReLU → Dropout, twice, plus
a skip connection) and a decoder, giving **one 128-d vector per row**. Every table gets
its own encoder, so different tables can have different numbers of columns.

In [4]:
model.eval()   # turns dropout off, so outputs are deterministic for the checks below
with torch.no_grad():
    x_dict = model.encoder(batch.tf_dict)
for nt, x in x_dict.items():
    print(f'{nt:22s} {batch[nt].tf.num_cols:2d} columns → {tuple(x.shape)}')

races                   4 columns → (33, 128)
qualifying              3 columns → (12, 128)
constructor_standings   4 columns → (0, 128)
standings               4 columns → (12, 128)
constructors            1 columns → (7, 128)
drivers                 1 columns → (4, 128)
constructor_results     2 columns → (0, 128)
circuits                3 columns → (0, 128)
results                11 columns → (12, 128)


## Step 4: relative-time encoding → $h^{(0)}$

For every row of a timed table, the model computes how many days before the seed's
prediction time it happened, encodes that number with sinusoids
(`torch_geometric.nn.PositionalEncoding`) and a linear layer, and **adds** it to the
row embedding. Static tables (`drivers`, `constructors`, `circuits`) get nothing.

In [5]:
seed_time = batch['drivers'].seed_time
days = (seed_time[batch['results'].batch] - batch['results'].time) / 86400
print('results rows: days before their seed time:', days[:6].tolist())

with torch.no_grad():
    rel_time = model.temporal_encoder(seed_time, batch.time_dict, batch.batch_dict)
    h0 = model.encode(batch, task.entity_table)   # encoder + time, exactly as in training
print('time encoding added to:', sorted(rel_time))
assert torch.allclose(h0['results'], x_dict['results'] + rel_time['results'])
assert torch.equal(h0['drivers'], x_dict['drivers'])
print('h0[results] = row embedding + time encoding ✓ ; drivers unchanged ✓')

results rows: days before their seed time: [1134.0, 1050.0, 777.0, 315.0, 222.0, 306.0]
time encoding added to: ['constructor_results', 'constructor_standings', 'qualifying', 'races', 'results', 'standings']
h0[results] = row embedding + time encoding ✓ ; drivers unchanged ✓


## Step 5: one GraphSAGE layer, by hand

Layer 1 updates every node from its sampled neighbours. For relation $r$ the
`SAGEConv` message into node $v$ is

$$m_{v,r} = W_{r,1} \sum_{u \in \mathcal N_r(v)} h_u + W_{r,2}\, h_v + b_r ,$$

`HeteroConv` sums $m_{v,r}$ over all relations that end at $v$'s type, and then each
node type has a LayerNorm and a ReLU:

$$h^{(1)}_v = \mathrm{ReLU}\Big(\mathrm{LN}\Big(\sum_r m_{v,r}\Big)\Big).$$

We recompute $h^{(1)}$ for **seed driver 0** with plain tensor operations and compare.

In [6]:
conv, norms = model.gnn.convs[0], model.gnn.norms[0]
v = 0
total = torch.zeros(128)
with torch.no_grad():
    for (src, rel, dst), ei in batch.edge_index_dict.items():
        if dst != 'drivers':
            continue
        sage = conv.convs[(src, rel, dst)]
        nbrs = ei[0, ei[1] == v]                       # neighbours of v under this relation
        m = sage.lin_l(h0[src][nbrs].sum(0)) + sage.lin_r(h0['drivers'][v])
        total += m
        print(f'{rel:22s} from {src:12s}: {len(nbrs)} neighbours')
    h1_by_hand = F.relu(norms['drivers'](total.unsqueeze(0)))[0]

    h1_model = {k: F.relu(norms[k](x)) for k, x in conv(h0, batch.edge_index_dict).items()}
print('\nby hand :', h1_by_hand[:6])
print('by model:', h1_model['drivers'][v, :6])
assert torch.allclose(h1_by_hand, h1_model['drivers'][v], atol=1e-5)
print('match ✓')

f2p_driverId           from qualifying  : 4 neighbours
f2p_driverId           from standings   : 4 neighbours
f2p_driverId           from results     : 4 neighbours

by hand : tensor([0.0000, 0.7966, 1.7478, 0.0000, 0.0000, 0.0000])
by model: tensor([0.0000, 0.7966, 1.7478, 0.0000, 0.0000, 0.0000])
match ✓


Note that a relation contributes $W_{r,2} h_v + b_r$ **even if $v$ has no neighbours
under it**, as long as the relation exists in the batch. A seed with an empty subgraph
(Walkthrough 1, Step 8) therefore still gets an update, built from its own features
only.

## Step 6: second layer and head → prediction

Layer 2 repeats the update, so information from 2 hops away reaches the seeds. The head
is a single linear layer applied to the seed drivers only (the first 4 `drivers` rows of
the batch). Its output is a *logit*; the sigmoid turns it into a probability.

In [7]:
with torch.no_grad():
    logits = model(batch, task.entity_table).view(-1)
print('logits       :', logits)
print('probabilities:', logits.sigmoid())
print('labels       :', batch['drivers'].y)

logits       : tensor([0.3338, 0.5586, 0.3179, 0.3365])
probabilities: tensor([0.5827, 0.6361, 0.5788, 0.5833])
labels       : tensor([0., 0., 1., 0.], dtype=torch.float64)


## Step 7: loss, by hand

Binary cross-entropy on the logits (`BCEWithLogitsLoss`):

$$\mathcal L = -\frac1B\sum_i \big[y_i \log\sigma(\hat y_i) + (1-y_i)\log(1-\sigma(\hat y_i))\big]$$

In [8]:
y = batch['drivers'].y.float()
p = logits.sigmoid()
by_hand = -(y * p.log() + (1 - y) * (1 - p).log()).mean()
by_torch = torch.nn.BCEWithLogitsLoss()(logits, y)
print(f'by hand {by_hand:.6f} | BCEWithLogitsLoss {by_torch:.6f}')
assert torch.allclose(by_hand, by_torch, atol=1e-6)
print('match ✓')

by hand 0.826770 | BCEWithLogitsLoss 0.826770
match ✓


## Step 8: one optimiser step

Back-propagate the loss and let Adam (lr 0.005) update every parameter once. On the
same batch, the loss should go down.

In [9]:
def batch_loss():  # loss on this batch with dropout off, so before/after are comparable
    model.eval()
    with torch.no_grad():
        return torch.nn.BCEWithLogitsLoss()(model(batch, task.entity_table).view(-1), y).item()

loss_before = batch_loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
model.train()   # dropout on, as in real training
loss = torch.nn.BCEWithLogitsLoss()(model(batch, task.entity_table).view(-1), y)
optimizer.zero_grad()
loss.backward()
grads = {n: p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None}
optimizer.step()

top = sorted(grads.items(), key=lambda kv: -kv[1])[:5]
print('largest gradient norms:')
for n, g in top:
    print(f'  {g:8.4f}  {n}')
no_grad = [n for n, p in model.named_parameters() if p.grad is None]
from collections import Counter
def group(name):  # e.g. 'encoder.encoders.circuits' or 'gnn.convs.1'
    parts = name.split('.')
    return '.'.join(parts[:3]) if parts[0] in ('encoder', 'gnn') else '.'.join(parts[:2])
print(f'\n{len(no_grad)} parameter tensors got no gradient (they did not affect this loss):')
for g, n in sorted(Counter(map(group, no_grad)).items()):
    print(f'  {n:3d}  {g}')
print(f'\nloss on this batch: before {loss_before:.4f} → after one step {batch_loss():.4f}')

largest gradient norms:


    2.5671  encoder.encoders.drivers.encoder.encoder_dict.timestamp.weight
    1.2700  encoder.encoders.drivers.decoder.2.weight
    0.9218  encoder.encoders.drivers.backbone.0.lin2.weight
    0.8560  encoder.encoders.drivers.backbone.1.lin1.weight
    0.8509  encoder.encoders.drivers.backbone.0.lin1.bias

268 parameter tensors got no gradient (they did not affect this loss):
   40  encoder.encoders.circuits
   42  encoder.encoders.constructor_results
   42  encoder.encoders.constructor_standings
   45  gnn.convs.0
   69  gnn.convs.1
   10  gnn.norms.0
   16  gnn.norms.1
    4  temporal_encoder.lin_dict

loss on this batch: before 0.8268 → after one step 0.5216


Parameters get no gradient when they could not influence the loss: the encoders of the three tables with no sampled rows, the first-layer convolutions of the relations that touch those tables, and the **second-layer** convolutions and norms of every node type except `drivers`. Only the seed drivers' final embeddings feed the head, so layer-2 updates of other node types are computed but never used.

## Step 9: evaluation on the validation split

Predictions for every validation row (real settings: fan-out `[128, 64]`, batch 512)
are scored with RelBench's evaluator. We compare the untrained model with one trained
for one epoch (3 batches of the 1,353 training rows).

In [10]:
def loader_for(split, shuffle):
    inp = get_node_train_table_input(table=task.get_table(split), task=task)
    return NeighborLoader(data, num_neighbors=[128, 64], time_attr='time',
                          input_nodes=inp.nodes, input_time=inp.time, transform=inp.transform,
                          batch_size=512, temporal_strategy='uniform', shuffle=shuffle)

@torch.no_grad()
def val_auc(model):
    model.eval()
    pred = torch.cat([model(b, task.entity_table).view(-1).sigmoid() for b in loader_for('val', False)])
    return task.evaluate(pred.numpy(), task.get_table('val'))['roc_auc']

seed_everything(0)
model = BaseModel(data, col_stats, channels=128, out_channels=1, num_layers=2, aggr='sum')
print(f'untrained   val ROC-AUC {val_auc(model):.4f}')

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
model.train()
for b in loader_for('train', True):
    loss = torch.nn.BCEWithLogitsLoss()(model(b, task.entity_table).view(-1), b['drivers'].y.float())
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f'  train batch: {b["drivers"].batch_size} seeds, loss {loss.item():.4f}')
print(f'after 1 epoch val ROC-AUC {val_auc(model):.4f}')

untrained   val ROC-AUC 0.6244


  train batch: 512 seeds, loss 0.9769


  train batch: 512 seeds, loss 0.5091


  train batch: 329 seeds, loss 0.4942


after 1 epoch val ROC-AUC 0.6371


One epoch is only 3 optimiser steps here, so it is not expected to move the validation
score much; the baseline trains for 10 epochs and keeps the best one. An untrained model
already scoring above 0.5 is not unusual: even random projections of the inputs (e.g.
whether a driver has any recent results) can rank drivers better than chance.

## Step 10: regression and the clipping range

For `driver-position` (average finishing position over the next 60 days) the loss is
L1, and at prediction time outputs are clipped to the 2nd–98th percentile range of the
training labels, so a few extreme predictions cannot dominate the error.

In [11]:
reg = dataset.load_task('driver-position')
labels = reg.get_table('train').df[reg.target_col]
lo, hi = np.percentile(labels, [2, 98])
print(f'train labels: min {labels.min():.2f}, max {labels.max():.2f}, mean {labels.mean():.2f}')
print(f'clipping range [2nd, 98th percentile] = [{lo:.2f}, {hi:.2f}]')
print('example:', torch.clamp(torch.tensor([-3.0, 7.5, 40.0]), lo, hi).tolist())

train labels: min 1.00, max 39.00, mean 13.90
clipping range [2nd, 98th percentile] = [2.00, 31.00]
example: [2.0, 7.5, 31.0]


## Recap

- `h⁽⁰⁾` = per-table ResNet over the column embeddings, plus a time encoding for timed
  tables. This is the input that the paper's filtering step (A) will gate.
- Each GraphSAGE layer sums one `SAGEConv` message per relation, then applies LayerNorm
  and ReLU. That sum over relations is where the paper's gated relations (step C) will
  plug in.
- The full baseline is this loop for 10 epochs, keeping the epoch with the best
  validation score (Plan 00).